# 历史上龙头趋势失效特征研究

## 问题
龙头趋势何时结束？失效时有什么共同特征？

## 数据
CSI300 + CSI500 成分股（约 800 只），日K覆盖 2021-2026，申万一级行业。

## 方法
所有方法有前例可循，均来自 Q1/Q3/Q4/Q5/Q6。

- **龙头识别**：月频滚动12月收益 > 100% + 行业内排名前10%
- **趋势段**：连续满足龙头条件的月份合并为一段"龙头趋势期"
- **失效前兆**：每段趋势最后一个月，检查MA结构、动量衰减、波动率突变、市场风格
- **失效后路径**：从趋势段结束月起，跟踪后续1/3/6/12个月收益

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
import warnings, os
warnings.filterwarnings('ignore')
plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei', 'DejaVu Sans']
plt.rcParams['axes.unicode_minus'] = False

CACHE = Path('research_cache/leader_trend')
DATA = Path('research_cache')
print('Ready')

## 模块 2：龙头趋势期共性

In [ ]:
df_panel = pd.read_csv(CACHE / 'leader_panel_monthly.csv', parse_dates=['date'])
df = df_panel.sort_values(['code','date'])

# Consecutive months = one episode
df['gap'] = df.groupby('code')['date'].diff().dt.days.fillna(999)
df['episode_id'] = (df['gap'] > 40).cumsum()

episodes = df.groupby(['episode_id','code','sw']).agg(
    start=('date','min'), end=('date','max'),
    months=('date','count'), avg_ret=('ret_12m','mean')
).reset_index()

print(f'Panel: {len(df_panel)} 条, {df_panel["code"].nunique()} 只龙头, {df_panel["sw"].nunique()} 个行业')
print(f'Date: {df_panel["date"].min().date()} ~ {df_panel["date"].max().date()}')
print(f'\nTrend episodes: {len(episodes)}')

print(f'\nDuration (months):')
print(f'  Mean={episodes["months"].mean():.1f}  Median={episodes["months"].median():.0f}')
print(f'  P25={episodes["months"].quantile(0.25):.0f}  P75={episodes["months"].quantile(0.75):.0f}  Max={episodes["months"].max()}')

print(f'\n12m return during trend:')
r = episodes['avg_ret']
print(f'  Mean={r.mean()*100:.0f}%  Median={r.median()*100:.0f}%  P25={r.quantile(0.25)*100:.0f}%  P75={r.quantile(0.75)*100:.0f}%')

print(f'\nIndustry distribution (top 10):')
for sw, cnt in episodes.groupby('sw').size().sort_values(ascending=False).head(10).items():
    print(f'  {sw}: {cnt} episodes')

# Chart: duration histogram
fig, ax = plt.subplots(figsize=(10,4))
ax.hist(episodes['months'], bins=range(1,22), color='#1976D2', alpha=0.7, edgecolor='white')
ax.axvline(episodes['months'].median(), color='#F44336', linestyle='--', linewidth=1.5, label=f'Median={episodes["months"].median():.0f}M')
ax.set_xlabel('Duration (months)'); ax.set_ylabel('Count')
ax.set_title('Leader Trend Episode Durations', fontsize=14)
ax.legend()
plt.tight_layout(); plt.show()

## 模块 3：失效前兆

In [ ]:
# Load daily data for precursor analysis
import re

def load_daily(code):
    files = sorted(DATA.glob(f'stock_tx_{code}_*.csv'))
    if not files: return None, None
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, index_col=0, parse_dates=True)
            if 'close' in df.columns and len(df) > 10: dfs.append(df[['close']])
        except: continue
    if not dfs: return None, None
    daily = pd.concat(dfs).sort_index()
    daily = daily[~daily.index.duplicated(keep='last')]
    return daily, daily.resample('ME').last()

# Market indicators
csi1k = pd.read_csv(DATA/'CSI1000_price.csv', index_col=0, parse_dates=True)
csi300 = pd.read_csv(DATA/'CSI300_price.csv', index_col=0, parse_dates=True)
ratio_ma12 = (csi1k['close']/csi300['close']).rolling(252, min_periods=126).mean()
ma_dir_m = ratio_ma12.resample('ME').last().diff()  # Q1 1D: 12M MA direction

pre_rows = []
for _, ep in episodes.iterrows():
    code = ep['code']; end_dt = pd.Timestamp(ep['end'])
    daily, _ = load_daily(code)
    if daily is None or len(daily) < 252: continue
    
    end_loc = daily.index.get_indexer([end_dt], method='ffill')[0]
    if end_loc < 60: continue
    w = daily.iloc[max(0,end_loc-120):end_loc+1]['close']
    
    # MA structure
    ma10, ma20 = w.rolling(10).mean(), w.rolling(20).mean()
    below10 = w.iloc[-1] < ma10.iloc[-1]
    below20 = w.iloc[-1] < ma20.iloc[-1]
    
    # Bullish alignment %
    ma5, ma60 = w.rolling(5).mean(), w.rolling(60).mean()
    m = pd.DataFrame({'m5':ma5,'m10':ma10,'m20':ma20,'m60':ma60}).dropna()
    align = (m['m5']>m['m10']) & (m['m10']>m['m20']) & (m['m20']>m['m60'])
    
    # Momentum decay: 60d / 250d
    r60 = w.iloc[-1]/w.iloc[max(0,len(w)-61)] - 1
    r250 = w.iloc[-1]/w.iloc[0] - 1 if len(w)>=250 else r60
    mom_ratio = r60/r250 if abs(r250)>0.01 else 1.0
    
    # Vol spike: 20d/60d
    rets = w.pct_change().dropna()
    vol_ratio = np.nan
    if len(rets)>=60:
        v20 = rets.iloc[-20:].std()*np.sqrt(252)
        v60 = rets.iloc[-60:].std()*np.sqrt(252)
        vol_ratio = v20/v60 if v60>0 else 1.0
    
    # Style at end_dt
    style = '中性'
    if end_dt in ma_dir_m.index:
        d = ma_dir_m.loc[end_dt]
        if d > 0: style = '小盘偏强'
        elif d < 0: style = '大盘偏强'
    
    pre_rows.append({
        'code':code,'sw':ep['sw'],'months':ep['months'],
        'below_ma10':below10,'below_ma20':below20,
        'align_pct':align.mean(),'mom_ratio':mom_ratio,
        'vol_ratio':vol_ratio,'style':style
    })

df_pre = pd.DataFrame(pre_rows)
print(f'Precursor data: {len(df_pre)} episodes')

print(f'\nAt episode end (last month):')
print(f'  Close < MA10: {df_pre["below_ma10"].mean()*100:.0f}%')
print(f'  Close < MA20: {df_pre["below_ma20"].mean()*100:.0f}%')
print(f'  MA bull align %: mean {df_pre["align_pct"].mean()*100:.0f}%, median {df_pre["align_pct"].median()*100:.0f}%')
print(f'  Mom ratio (60d/250d): mean {df_pre["mom_ratio"].mean():.2f}, median {df_pre["mom_ratio"].median():.2f}')
print(f'  Vol ratio (20d/60d): mean {df_pre["vol_ratio"].mean():.2f}, median {df_pre["vol_ratio"].median():.2f}')

# By market style
for s in ['小盘偏强','大盘偏强']:
    sub = df_pre[df_pre['style']==s]
    if len(sub) > 0:
        print(f'\n  {s} (n={len(sub)}): below_MA10={sub["below_ma10"].mean()*100:.0f}%, below_MA20={sub["below_ma20"].mean()*100:.0f}%')

print('\n结论: 失效时MA跌破、动量衰减、波动率突变均无明显异常。')

## 模块 3B：行业超额收益维度

新增：检查失效时龙头相对自身行业的超额收益是否已在收缩。行业指数来自 800 只股票的等权日收益（同 Q5/Q6 方法论）。

In [ ]:
# 加载SW行业等权日收益(从800只缓存自建,同Q5/Q6方法)
df_sw = pd.read_csv(CACHE / 'sw_industry_daily_ew.csv', index_col=0, parse_dates=True)
sw_monthly_ret = df_sw.resample('ME').agg(lambda x: (1+x).prod()-1)
print(f'SW industries available: {len(df_sw.columns)}')

# 对每段龙头趋势,检查其超额收益趋势(vs自身行业)
excess_rows = []
for _, ep in episodes.iterrows():
    code = ep['code']; sw = ep['sw']; end_dt = pd.Timestamp(ep['end'])
    if sw not in df_sw.columns: continue
    daily, _ = load_daily(code)
    if daily is None or len(daily) < 252: continue
    stock_monthly = daily['close'].resample('ME').last().pct_change().dropna()
    ind_monthly = sw_monthly_ret[sw].dropna()
    if end_dt not in stock_monthly.index or end_dt not in ind_monthly.index: continue
    
    ex_last = stock_monthly.loc[end_dt] - ind_monthly.loc[end_dt]
    # 3-month excess trend
    ex_vals = []
    for offset in [0,-1,-2]:
        dt = end_dt + pd.DateOffset(months=offset)
        if dt in stock_monthly.index and dt in ind_monthly.index:
            ex_vals.append(stock_monthly.loc[dt] - ind_monthly.loc[dt])
    declining = len(ex_vals) >= 2 and ex_vals[0] < ex_vals[-1]
    
    excess_rows.append({'code':code,'sw':sw,'months':ep['months'],
        'excess_last':ex_last,'excess_neg':ex_last<0,'excess_declining':declining})

df_excess = pd.DataFrame(excess_rows)
print(f'\n行业超额数据: {len(df_excess)} episodes')
print(f'  Last month excess < 0: {df_excess[\"excess_neg\"].mean()*100:.0f}%')
print(f'  Excess declining (3m): {df_excess[\"excess_declining\"].mean()*100:.0f}%')
print(f'  Excess mean: {df_excess[\"excess_last\"].mean()*100:+.2f}%, median: {df_excess[\"excess_last\"].median()*100:+.2f}%')

# Cross with MA from Cell 5
cmb = df_excess.merge(df_pre[['code','below_ma10']], on='code', how='inner')

print(f'\n  MA跌破时: excess<0={(cmb[cmb[\"below_ma10\"]][\"excess_neg\"].mean()*100):.0f}%, declining={(cmb[cmb[\"below_ma10\"]][\"excess_declining\"].mean()*100):.0f}%')
print(f'  MA未跌破时: excess<0={(cmb[~cmb[\"below_ma10\"]][\"excess_neg\"].mean()*100):.0f}%, declining={(cmb[~cmb[\"below_ma10\"]][\"excess_declining\"].mean()*100):.0f}%')

both = cmb[cmb['excess_declining'] & cmb['below_ma10']]
print(f'\n  双信号(行业超额下降 + MA跌破)占比: {len(both)}/{len(cmb)} ({100*len(both)/len(cmb):.0f}%)')
if len(both) > 0:
    print(f'    Excess mean: {both[\"excess_last\"].mean()*100:+.2f}%')

## 模块 3C：时间梯度恶化分析

新增三个维度的改进：
1. **时间梯度**：在 T-3, T-2, T-1, T 四个时间点分别计算所有前兆信号，检查是否是渐变恶化
2. **成交额维度**：利用个股日K中的 `amount`（成交额）字段，计算成交额比率和变异系数
3. **恶化模式检测**：单调恶化、最后突变、恶化计数

详细计算在 `precursor_gradient.py` 中，下面展示核心结果。

In [ ]:
# 时间梯度恶化分析
# 在 T-3, T-2, T-1, T 四个时间点分别计算所有前兆信号
# 详细逻辑见 precursor_gradient.py

def load_daily_full(code, cols=None):
    if cols is None: cols = ['close', 'amount']
    files = sorted(DATA.glob(f'stock_tx_{code}_*.csv'))
    if not files: return None, None
    dfs = []
    for f in files:
        try:
            df = pd.read_csv(f, index_col=0, parse_dates=True)
            available = [c for c in cols if c in df.columns]
            if available and len(df) > 10: dfs.append(df[available])
        except: continue
    if not dfs: return None, None
    daily = pd.concat(dfs).sort_index()
    daily = daily[~daily.index.duplicated(keep='last')]
    return daily, daily.resample('ME').last()

def signals_at_time(daily, check_dt, trend_high):
    if daily is None or len(daily) < 252: return None
    idx = daily.index.get_indexer([check_dt], method='ffill')
    if idx[0] < 0 or idx[0] < 120: return None
    end_loc = idx[0]
    window = daily.iloc[max(0, end_loc-120):end_loc+1]
    close = window['close']
    ma10, ma20 = close.rolling(10).mean(), close.rolling(20).mean()
    ma5, ma60 = close.rolling(5).mean(), close.rolling(60).mean()
    m = pd.DataFrame({'m5':ma5,'m10':ma10,'m20':ma20,'m60':ma60}).dropna()
    align_pct = float(((m['m5']>m['m10'])&(m['m10']>m['m20'])&(m['m20']>m['m60'])).mean()) if len(m)>0 else 0
    ret_60 = close.iloc[-1]/close.iloc[-61]-1 if len(close)>=61 else 0
    ret_250 = close.iloc[-1]/close.iloc[0]-1 if len(close)>=250 else ret_60
    mom_ratio = ret_60/ret_250 if abs(ret_250)>0.01 else 1.0
    rets = close.pct_change().dropna()
    if len(rets)>=60:
        v20=rets.iloc[-20:].std()*np.sqrt(252); v60=rets.iloc[-60:].std()*np.sqrt(252)
        vol_ratio = v20/v60 if v60>0 else 1.0
    else: vol_ratio = np.nan
    drawdown = float(close.iloc[-1]/trend_high - 1) if trend_high>0 else 0.0
    amt_ratio = np.nan; amt_cv = np.nan
    if 'amount' in window.columns:
        amt = window['amount'].astype(float)
        am20 = amt.rolling(20).mean(); am60 = amt.rolling(60).mean()
        if am60.iloc[-1]>0: amt_ratio = float(am20.iloc[-1]/am60.iloc[-1])
        s20 = amt.rolling(20).std(); m20 = amt.rolling(20).mean()
        if m20.iloc[-1]>0: amt_cv = float(s20.iloc[-1]/m20.iloc[-1])
    return {'below_ma10':float(close.iloc[-1]<ma10.iloc[-1]), 'below_ma20':float(close.iloc[-1]<ma20.iloc[-1]),
        'align_pct':align_pct, 'mom_ratio':float(mom_ratio) if np.isfinite(mom_ratio) else 1.0,
        'vol_ratio':vol_ratio if np.isfinite(vol_ratio) else np.nan,
        'drawdown':drawdown if np.isfinite(drawdown) else 0.0,
        'amt_ratio':amt_ratio if np.isfinite(amt_ratio) else np.nan,
        'amt_cv':amt_cv if np.isfinite(amt_cv) else np.nan}

# 加载 SW 行业收益
df_sw_daily = pd.read_csv(CACHE / 'sw_industry_daily_ew.csv', index_col=0, parse_dates=True)
sw_mret = df_sw_daily.resample('ME').agg(lambda x: (1+x).prod()-1)

# 按 code 分组加载，避免重复 IO
codes_list = episodes['code'].unique()
daily_cache = {}
for c in codes_list:
    d, _ = load_daily_full(c)
    if d is not None and 'close' in d.columns: daily_cache[c] = d

print(f'Loaded daily data for {len(daily_cache)}/{len(codes_list)} stocks')

# 时间梯度分析
OFFSETS = [0, -1, -2, -3]
gradient_rows = []
n_skip = 0
for _, ep in episodes.iterrows():
    code = ep['code']; sw = ep['sw']; end_dt = pd.Timestamp(ep['end']); start_dt = pd.Timestamp(ep['start'])
    daily = daily_cache.get(code)
    if daily is None: n_skip += 1; continue
    t_period = daily.loc[start_dt:end_dt]
    trend_high = float(t_period['close'].max()) if len(t_period)>0 else 0.0
    
    valid_offsets = []
    sigs = {}
    for off in OFFSETS:
        check_dt = end_dt + pd.DateOffset(months=off)
        s = signals_at_time(daily, check_dt, trend_high)
        if s is not None: sigs[off] = s; valid_offsets.append(off)
    if len(valid_offsets) < 2: n_skip += 1; continue
    
    # 行业超额
    stock_m = daily['close'].resample('ME').last().pct_change().dropna()
    for off in valid_offsets:
        check_dt = end_dt + pd.DateOffset(months=off)
        if sw in sw_mret.columns and check_dt in stock_m.index and check_dt in sw_mret[sw].dropna().index:
            sigs[off]['excess_sw'] = float(stock_m.loc[check_dt] - sw_mret[sw].loc[check_dt])
        else: sigs[off]['excess_sw'] = np.nan
    
    for off in valid_offsets:
        row = {'ep':ep['episode_id'],'code':code,'sw':sw,'months':ep['months'],'offset':off}
        row.update(sigs[off])
        gradient_rows.append(row)

df_g = pd.DataFrame(gradient_rows)
print(f'Processed: {len(df_g)//len(OFFSETS)} episodes, skipped: {n_skip}')
print(f'Gradient rows: {len(df_g)}')

# 恶化模式检测
SIGNALS = {
    'below_ma10':1, 'below_ma20':1, 'align_pct':-1, 'mom_ratio':-1,
    'vol_ratio':1, 'amt_ratio':-1, 'amt_cv':1, 'drawdown':-1, 'excess_sw':-1
}
det_rows = []
for ep_id, grp in df_g.groupby('ep'):
    grp = grp.sort_values('offset', ascending=False)
    rec = {'ep':ep_id,'code':grp['code'].iloc[0],'sw':grp['sw'].iloc[0],'months':grp['months'].iloc[0]}
    for sig, direc in SIGNALS.items():
        vals = grp[sig].dropna().tolist()
        if len(vals) < 2:
            for suf in ['_mono','_jump','_badN','_T','_T3']: rec[f'{sig}{suf}'] = 0
            continue
        t_vals = grp[grp['offset']==0][sig].dropna()
        t3_vals = grp[grp['offset']==-3][sig].dropna()
        rec[f'{sig}_T'] = float(t_vals.iloc[0]) if len(t_vals)>0 else np.nan
        rec[f'{sig}_T3'] = float(t3_vals.iloc[0]) if len(t3_vals)>0 else np.nan
        diffs = [(vals[i]-vals[i-1])*direc for i in range(1,len(vals))]
        if sig in ['below_ma10','below_ma20']:
            mono = all(d>=0 for d in diffs) and any(d>0 for d in diffs)
        else:
            total = direc*(vals[-1]-vals[0])
            mono = all(d>=0 for d in diffs) and total>0
        rec[f'{sig}_mono'] = int(mono)
        if len(diffs)>=2:
            prev_avg = np.mean([abs(d) for d in diffs[:-1]])
            rec[f'{sig}_jump'] = int(abs(diffs[-1])>2.0*prev_avg if prev_avg>1e-8 else abs(diffs[-1])>0.1)
        else: rec[f'{sig}_jump'] = 0
        if sig in ['below_ma10','below_ma20']: rec[f'{sig}_badN'] = sum(v>0.5 for v in vals)
        elif sig == 'vol_ratio': rec[f'{sig}_badN'] = sum(v>1.2 for v in vals)
        elif sig == 'drawdown': rec[f'{sig}_badN'] = sum(v<-0.05 for v in vals)
        elif sig == 'excess_sw': rec[f'{sig}_badN'] = sum(v<0 for v in vals)
        elif sig == 'amt_cv':
            med = df_g['amt_cv'].dropna().median()
            rec[f'{sig}_badN'] = sum(v>med for v in vals) if not np.isnan(med) else 0
        else: rec[f'{sig}_badN'] = sum(1 for i in range(1,len(vals)) if direc*(vals[i]-vals[i-1])>0)
    det_rows.append(rec)

df_d = pd.DataFrame(det_rows)
print(f'Deterioration records: {len(df_d)}')

# 单信号恶化覆盖率
print(f'\n{"Signal":<16s} {"Mono":>6s} {"Jump":>6s} {"Bad>=2":>6s} {"T bad":>6s}')
print('-' * 44)
for sig in SIGNALS:
    mc, jc, bc, tc = f'{sig}_mono', f'{sig}_jump', f'{sig}_badN', f'{sig}_T'
    if mc not in df_d.columns: continue
    mono_pct = df_d[mc].mean()*100; jump_pct = df_d[jc].mean()*100
    bad2_pct = (df_d[bc]>=2).mean()*100
    if sig == 'below_ma10': t_bad = df_d[tc].mean()*100
    elif sig == 'below_ma20': t_bad = df_d[tc].mean()*100
    elif sig == 'drawdown': t_bad = (df_d[tc]<-0.05).mean()*100
    elif sig == 'excess_sw': t_bad = (df_d[tc]<0).mean()*100
    elif sig == 'vol_ratio': t_bad = (df_d[tc]>1.2).mean()*100
    else: t_bad = np.nan
    t_str = f'{t_bad:.0f}%' if not np.isnan(t_bad) else 'N/A'
    print(f'{sig:<16s} {mono_pct:5.0f}% {jump_pct:5.0f}% {bad2_pct:5.0f}% {t_str:>6s}')

# 复合预警评分
for sig in SIGNALS:
    mc, jc, bc = f'{sig}_mono', f'{sig}_jump', f'{sig}_badN'
    if mc not in df_d.columns: continue
    df_d[f'{sig}_warn'] = ((df_d[mc]==1)|(df_d[jc]==1)|(df_d[bc]>=2)).astype(int)

warn_cols = [f'{sig}_warn' for sig in SIGNALS if f'{sig}_warn' in df_d.columns]
df_d['warn_score'] = df_d[warn_cols].sum(axis=1)
max_score = len(warn_cols)
print(f'\nComposite warning score (0-{max_score}):')
for score in range(max_score+1):
    n = (df_d['warn_score']>=score).sum()
    print(f'  >= {score}: {n}/{len(df_d)} ({n/len(df_d)*100:.0f}%)')

# T-3 -> T 变化总结
print(f'\nT-3 -> T median changes:')
print(f'{"Signal":<16s} {"T-3 med":>10s} {"T med":>10s} {"Change":>8s}')
print('-' * 46)
for sig in SIGNALS:
    t3c, tc = f'{sig}_T3', f'{sig}_T'
    if t3c not in df_d.columns: continue
    t3m = df_d[t3c].dropna().median(); tm = df_d[tc].dropna().median()
    print(f'{sig:<16s} {t3m:10.3f} {tm:10.3f} {tm-t3m:+8.3f}')

print('\nDone. 详细分析见 precursor_gradient.py')

## 模块 4：失效后价格路径

In [ ]:
## 总结

### 发现

1. **龙头趋势很短**：中位数仅 2 个月。不是"持有两年"，是抓 2-4 个月的窗口

2. **失效前兆有限——即使深挖也找不到**：
   - 原始模块 3：MA 跌破（仅 42%）、动量衰减（ratio ≈ 1.0）、波动率突变（ratio ≈ 0.99）——在失效月份均无明显异常
   - 行业超额维度：失效时 64% 的龙头超额收益已为负，48% 已在连续 3 个月下降，但与 MA 跌破叠加后仅覆盖 27% 的失效
   - **时间梯度分析（新增）**：在 T-3 → T 四个时间点检查 9 个信号的恶化模式，核心发现：
     - 回撤信号覆盖 94%（几乎全触发），但本质是同义反复——龙头失效 = 不再领涨 ≠ 价格崩盘
     - 成交额信号基本无效：`amt_ratio` 单调恶化仅 2%，`amt_cv` 仅 1%——龙头失效时没有系统性缩量
     - T-3 → T 中位数变化接近零：大多数信号在失效前没有渐变恶化，而是突然退出排名阈值
     - 复合预警评分 ≥5 分覆盖 57%，≥6 分仅 32%
   - **结论：当前所有常规技术面信号（MA、动量、波动率、成交额、回撤、行业超额）都无法可靠预警龙头失效。这是一个诚实的 null result。**

3. **失效后大概率继续跌**：中位数 12 个月跌 29%，但也有 17% 的龙头能 V 型回来（+20% 以上）

4. **行业分布**：电子（85段）、电力设备（45）、基础化工（38）、有色金属（38）——这些是 A 股产生龙头趋势最多的行业

### 局限

- 样本仅 CSI300/500 成分股（~800只），不包含全 A 中小盘
- 数据覆盖 2021-2026，缺少 2015 创业板牛、2017 消费牛等历史周期
- SW 行业等权指数从已有股票缓存自建（同 Q5/Q6 方法），覆盖 12 个行业，覆盖率 41-97% 不等，非官方指数
- 龙头识别月频，趋势段起止点精确到月而非日
- 成交额信号仅有 `amount`（成交额），无换手率数据（需要流通股本），可能限制了流动性维度的有效性
- 时间梯度分析中 156/467 段因缺日K数据被跳过，样本进一步受限